# 08 - School District Spatial Join

### Week 6 (deferred):
- Spatially join property coordinates to CA School District Areas 2024-25
- Add district-level attributes as features
- Retrain LightGBM and compare against 07 results

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import lightgbm as lgb
import re

train_df = pd.read_csv("data/train_final.csv")
test_df = pd.read_csv("data/test_final.csv")

districts = gpd.read_file('data/ca_school_districts/DistrictAreas2425.shp')
districts = districts.to_crs('EPSG:4326')  # match lat/long

# keep only what we need
keep = ['DistrictNa', 'DistrictTy', 'EnrollTota', 'SEDpct', 'ELpct', 'geometry']
districts = districts[keep]

print('Districts:', districts.shape)
print(districts['DistrictTy'].value_counts())

Districts: (937, 6)
DistrictTy
Elementary    516
Unified       345
High           76
Name: count, dtype: int64


In [2]:
def join_districts(df):
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
        crs='EPSG:4326'
    )
    joined = gpd.sjoin(gdf, districts, how='left', predicate='within')
    
    # a property can land in multiple districts (Elementary + High overlap Unified)
    # priority: Unified > Elementary > High
    priority = {'Unified': 0, 'Elementary': 1, 'High': 2}
    joined['_prio'] = joined['DistrictTy'].map(priority)
    joined = joined.sort_values('_prio').groupby(level=0).first()
    
    return joined.drop(columns=['geometry', 'index_right', '_prio'])

train_joined = join_districts(train_df)
test_joined = join_districts(test_df)

print('Train:', train_joined.shape)
print('Unmatched (no district):', train_joined['DistrictNa'].isna().sum())
print()
print(train_joined['DistrictTy'].value_counts())
print()
print(train_joined[['DistrictNa', 'EnrollTota', 'SEDpct', 'ELpct']].head())


Train: (230240, 4006)
Unmatched (no district): 98

DistrictTy
Unified       173490
Elementary     56652
Name: count, dtype: int64

             DistrictNa  EnrollTota  SEDpct  ELpct
0      Monrovia Unified      4920.0    62.2    0.4
1  Palm Springs Unified     20008.0    94.8    1.0
2    Capistrano Unified     46660.0    36.2    0.2
3     San Diego Unified    113787.0    60.5    0.2
4      Redlands Unified     19572.0    62.3    0.5


In [3]:
district_feats = ['EnrollTota', 'SEDpct', 'ELpct']

# fill unmatched with train medians
for c in district_feats:
    med = train_joined[c].median()
    train_joined[c] = train_joined[c].fillna(med)
    test_joined[c] = test_joined[c].fillna(med)

drop_from_features = ['ClosePrice', 'ClosePrice_log', 'CloseDate', 'CloseYearMonth',
                      'DistrictNa', 'DistrictTy']

X_train = train_joined.drop(columns=[c for c in drop_from_features if c in train_joined.columns])
X_test = test_joined.drop(columns=[c for c in drop_from_features if c in test_joined.columns])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_joined['ClosePrice_log']
y_test = test_joined['ClosePrice_log']
actual_price = np.exp(y_test)

# sanitize column names for lightgbm
clean_cols = [re.sub(r'[^A-Za-z0-9_]+', '_', str(c)) for c in X_train.columns]
seen, final_cols = {}, []
for c in clean_cols:
    if c in seen:
        seen[c] += 1
        final_cols.append(f"{c}_{seen[c]}")
    else:
        seen[c] = 0
        final_cols.append(c)
X_train.columns = final_cols
X_test.columns = final_cols

print('Feature count:', X_train.shape[1])

lgb_school = lgb.LGBMRegressor(
    num_leaves=127, learning_rate=0.05, n_estimators=1000, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1
)
lgb_school.fit(X_train, y_train)

pred = lgb_school.predict(X_test)
pred_price = np.exp(pred)

r2 = r2_score(y_test, pred)
mape = mean_absolute_percentage_error(actual_price, pred_price)
mdape = np.median(np.abs((actual_price - pred_price) / actual_price))
print(f"LightGBM + school districts — R²: {r2:.4f}, MAPE: {mape:.4f}, MdAPE: {mdape:.4f}")

Feature count: 4000
LightGBM + school districts — R²: 0.9431, MAPE: 0.1143, MdAPE: 0.0757


In [4]:
imp = pd.Series(lgb_school.feature_importances_, index=X_train.columns)
print(imp[['EnrollTota', 'SEDpct', 'ELpct']].sort_values(ascending=False))
print()
print('Rank among all features:')
ranks = imp.rank(ascending=False)
print(ranks[['EnrollTota', 'SEDpct', 'ELpct']])
print('Total features:', len(imp))

EnrollTota    6003
SEDpct        5736
ELpct         2790
dtype: int32

Rank among all features:
EnrollTota     7.0
SEDpct         9.0
ELpct         14.0
dtype: float64
Total features: 4000


# 08 - School District Spatial Join — Summary

**What this notebook does:**
1. Loads CA School District Areas 2024-25 boundaries (937 polygons, EPSG:3857 → reprojected to EPSG:4326)
2. Spatially joins each property's coordinates to its containing district via point-in-polygon
3. Resolves overlapping districts by priority (Unified > Elementary > High)
4. Adds three district-level attributes as features and retrains the tuned LightGBM

**Join results:** 230,240 train properties matched, 98 unmatched (0.04%, filled with train medians). Distribution: 173,490 Unified, 56,652 Elementary.

**Features added:** `EnrollTota` (total district enrollment), `SEDpct` (% socioeconomically disadvantaged), `ELpct` (% English learners)

**Results:**

| Model | R² | MAPE | MdAPE |
|---|---|---|---|
| LightGBM (tuned) | 0.9400 | 0.1176 | 0.0776 |
| **LightGBM + school districts** | **0.9431** | **0.1143** | **0.0757** |

**Takeaway:** The first feature addition in this project to produce a measurable gain. Feature importance ranks the three district columns 7th, 9th, and 14th out of 4,000 — the top 0.5%.

`SEDpct` acts as a neighborhood affluence proxy and `EnrollTota` distinguishes dense urban districts from small rural ones. Both capture economic context that property-level features can't.

This also explains why the KMeans geo clustering in 06 failed: arbitrary geometric clusters carry no information beyond position, while school district boundaries follow real community lines and come with attached demographics that correlate directly with price.

**Design decision:** District identity was added as numeric attributes rather than one-hot encoded district names. One-hot encoding 937 districts would have added ~900 columns to an already 4,000-wide matrix; three numeric columns achieved the gain without the dimensionality cost.